
This notebook has been revised into a **6 exercises / 6 points** format. The whiteboard exercises do not ask for abstract theory only; they require students to **run the algorithm by hand** or **adapt the algorithm to fit the problem**.

| Exercise | Format | Main topic | Points |
|---|---|---|---:|
| 1 | Whiteboard algorithm | Run the RL loop + compute returns + adjust the policy | 1 |
| 2 | Code | Value Iteration on FrozenLake | 1 |
| 3 | Whiteboard algorithm | Run Bellman backup / Q-learning backup by hand | 1 |
| 4 | Code | Tabular Q-learning | 1 |
| 5 | Code | DQN | 1 |
| 6 | Whiteboard adaptation + short code | REINFORCE / Actor-Critic | 1 |

The required knowledge scope follows the original slides: MDP, policy, return, Bellman equation, Value Iteration, Q-learning, DQN, REINFORCE, Actor-Critic.



# Required knowledge summary

Use this section as the minimum theory checklist before attempting the exercises.

## 1. MDP and RL loop

- A Markov Decision Process is \((S, A, P, R, \gamma)\): states, actions, transition dynamics, rewards, and discount factor.
- At each step the agent observes \(s_t\), chooses \(a_t\), receives \(r_{t+1}\), and moves to \(s_{t+1}\).
- The Markov property means the next state distribution depends on the current state and action, not the full history.

## 2. Policy, return, and value functions

- A policy \(\pi(a|s)\) tells the agent how to choose actions. It can be deterministic or stochastic.
- The discounted return is \(G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + ...\).
- State value: \(V^\pi(s) = \mathbb{E}_\pi[G_t | s_t=s]\).
- Action value: \(Q^\pi(s,a) = \mathbb{E}_\pi[G_t | s_t=s, a_t=a]\).
- A greedy policy chooses \(\arg\max_a Q(s,a)\) or uses one-step lookahead from \(V(s)\).

## 3. Bellman equations and dynamic programming

- Bellman expectation backup: value equals immediate reward plus discounted expected future value.
- Bellman optimality backup for values: \(V^*(s)=\max_a \sum_{s'} P(s'|s,a)[R(s,a,s') + \gamma V^*(s')]\).
- Value Iteration repeatedly applies the optimality backup, then extracts the greedy policy.
- Synchronous updates use a copy of the old value table; in-place updates reuse newly updated values immediately.

## 4. Model-free control with Q-learning

- Q-learning does not need \(P\) or \(R\); it learns from sampled transitions.
- Update: \(Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma \max_{a'} Q(s',a') - Q(s,a)]\).
- The term in brackets is the temporal-difference error.
- Epsilon-greedy explores with probability \(\epsilon\) and otherwise chooses the current best action.

## 5. Function approximation and DQN

- DQN replaces the Q-table with a neural network \(Q_\theta(s,a)\), useful when states are large or continuous.
- Experience replay samples old transitions to reduce correlation between updates.
- A target network computes a more stable target \(r + \gamma \max_{a'} Q_{\theta^-}(s',a')\).
- The DQN loss is usually mean squared error between predicted Q-values and Bellman targets.

## 6. Policy gradients and Actor-Critic

- REINFORCE optimizes the policy directly using \(-\log \pi_\theta(a_t|s_t) G_t\) as the loss contribution.
- High return increases the probability of chosen actions; low return decreases it.
- A baseline or critic estimates value and reduces variance, often using advantage \(A(s,a)=Q(s,a)-V(s)\).
- Actor-Critic combines a policy network (actor) with a value estimator (critic).

## 7. What you should be able to do by hand

- Trace one trajectory and compute its discounted returns.
- Run one Bellman backup or one Q-learning update with numbers.
- Explain why exploration is needed when the current greedy action is wrong.
- Read a learned value table, Q-table, or neural-network target and identify the induced policy.



# Setup


In [ ]:

# If running on Colab and some libraries are missing:
# !pip install gymnasium[classic-control,toy-text] matplotlib numpy torch


In [ ]:

import random
import numpy as np
import matplotlib.pyplot as plt
from collections import deque, namedtuple

import gymnasium as gym

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    import torch.nn.functional as F
except ImportError:
    torch = None
    print("Torch is not installed. The DQN / REINFORCE exercises require PyTorch.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)

def reset_env(env):
    out = env.reset(seed=SEED)
    return out[0] if isinstance(out, tuple) else out

def step_env(env, action):
    out = env.step(action)
    if len(out) == 5:
        next_state, reward, terminated, truncated, info = out
        return next_state, reward, terminated or truncated, info
    return out



# Exercise 1 - Whiteboard algorithm: Run the RL loop and compute returns

Given the following 3x3 grid:

```text
S  .  .
.  H  .
.  .  G
```

Legend:

- `S`: start state at `(0,0)`
- `H`: hole at `(1,1)`; falling into it ends the episode, reward = `-1`
- `G`: goal at `(2,2)`; reaching it ends the episode, reward = `+1`
- Normal cells have reward = `0`
- Actions: `UP`, `DOWN`, `LEFT`, `RIGHT`
- If the agent moves outside the grid boundary, it stays in place and reward = `0`

Given the fixed policy:

```text
From every normal cell:
if moving RIGHT is possible, move RIGHT;
otherwise, move DOWN.
```

## Requirements

1. Write the trajectory from `S` under the policy above.
2. Clearly write each step: `state -> action -> next_state -> reward`.
3. Compute the return \(G_0\) with \(\gamma = 1\).
4. Compute the return \(G_0\) with \(\gamma = 0.9\).
5. Is this policy good? If not, manually adjust the policy so it reaches the goal while avoiding the hole.


## Student answer section

Write your solution / calculations here.



# Exercise 2 - Code: Value Iteration on FrozenLake

This exercise tests **known MDPs**.

Because FrozenLake has the transition model `P(s'|s,a)`, we can use the Bellman optimality equation:


$$
V^*(s) = \max_a \sum_{s'} P(s' \mid s,a)\Bigl[r(s,a,s') + \gamma V^*(s')\Bigr]
$$


## Requirements

Complete the `value_iteration` function.

Then print:

1. `V*`
2. the corresponding greedy policy


In [ ]:

def value_iteration(env, gamma=0.99, theta=1e-8, max_iter=10_000):
    nS = env.observation_space.n
    nA = env.action_space.n
    P = env.unwrapped.P

    V = np.zeros(nS)

    for it in range(max_iter):
        delta = 0
        new_V = V.copy()

        for s in range(nS):
            action_values = []

            for a in range(nA):
                q_sa = 0

                # TODO:
                # Iterate through possible transitions from state s and action a.
                # P[s][a] contains tuples: (prob, next_state, reward, done)
                for prob, next_s, reward, done in P[s][a]:
                    q_sa += prob * (reward + gamma * V[next_s] * (not done))

                action_values.append(q_sa)

            best_value = max(action_values)
            new_V[s] = best_value
            delta = max(delta, abs(V[s] - best_value))

        V = new_V

        if delta < theta:
            print(f"Converged after {it + 1} iterations")
            break

    policy = np.zeros(nS, dtype=int)

    for s in range(nS):
        action_values = []

        for a in range(nA):
            q_sa = 0
            for prob, next_s, reward, done in P[s][a]:
                q_sa += prob * (reward + gamma * V[next_s] * (not done))
            action_values.append(q_sa)

        policy[s] = int(np.argmax(action_values))

    return V, policy


env = gym.make("FrozenLake-v1", is_slippery=True)
V, policy = value_iteration(env)

print("V*:")
print(V.reshape(4, 4))

print("\nGreedy policy:")
print(policy.reshape(4, 4))

print("\nAction mapping: 0=LEFT, 1=DOWN, 2=RIGHT, 3=UP")
env.close()



# Exercise 3 - Whiteboard algorithm: Run Bellman backup and Q backup by hand

Given a small MDP with 3 states:

```text
s0: start
s1: intermediate
sT: terminal
```

There are 2 actions at `s0`:

```text
action A: go to s1, reward = 0
action B: go to sT, reward = 1
```

At `s1`, there is only 1 action:

```text
action C: go to sT, reward = 3
```

`sT` is terminal, so:

$$
V(s_T) = 0
$$

Given \(\gamma = 0.9\).

## Requirement A - Run a Value backup

Assume initially:

$$
V_0(s_0)=0,\quad V_0(s_1)=0,\quad V_0(s_T)=0
$$

Compute:

1. \(V_1(s_1)\)
2. \(V_1(s_0)\)
3. What is the best action at `s0` after this backup?

## Requirement B - Run a Q-learning backup

Assume currently:

```text
Q(s0,A) = 0
Q(s0,B) = 0
Q(s1,C) = 0
```

If the agent takes the transition:

```text
s0 --A, reward=0--> s1
```

Update \(Q(s0,A)\) with:

```text
alpha = 0.5
gamma = 0.9
```

Use the Q-learning update:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha
\Bigl[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\Bigr]
$$


## Student answer section

Write your solution / calculations here.



# Exercise 4 - Code: Tabular Q-learning

This exercise tests **unknown MDPs**.

Do not use the transition model directly anymore. The agent learns by interacting with the environment.

Q-learning update:


$$
Q(s,a) \leftarrow Q(s,a) + \alpha \Bigl[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\Bigr]
$$


## Requirements

Complete / run the Q-learning code:

1. Train on FrozenLake.
2. Plot the moving average reward.
3. Print the learned policy.
4. Explain the role of epsilon-greedy.


In [ ]:

def epsilon_greedy(Q, state, epsilon, nA):
    if random.random() < epsilon:
        return random.randrange(nA)
    return int(np.argmax(Q[state]))

def train_q_learning(
    episodes=20_000,
    alpha=0.1,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.9995,
    is_slippery=True
):
    env = gym.make("FrozenLake-v1", is_slippery=is_slippery)

    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA))

    epsilon = epsilon_start
    episode_rewards = []

    for ep in range(episodes):
        state = reset_env(env)
        total_reward = 0

        for t in range(200):
            action = epsilon_greedy(Q, state, epsilon, nA)
            next_state, reward, done, info = step_env(env, action)

            # TODO: Q-learning target
            target = reward + gamma * np.max(Q[next_state]) * (not done)

            # TODO: TD error
            td_error = target - Q[state, action]

            # TODO: update Q
            Q[state, action] += alpha * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        episode_rewards.append(total_reward)

    env.close()
    return Q, episode_rewards

Q, rewards = train_q_learning()

window = 500
moving_avg = np.convolve(rewards, np.ones(window) / window, mode="valid")

plt.figure()
plt.plot(moving_avg)
plt.title("Q-learning on FrozenLake")
plt.xlabel("Episode")
plt.ylabel("Moving average reward")
plt.show()

print("Learned greedy policy:")
print(np.argmax(Q, axis=1).reshape(4, 4))
print("\nAction mapping: 0=LEFT, 1=DOWN, 2=RIGHT, 3=UP")



# Exercise 5 - Code: Deep Q-learning / DQN on CartPole

This exercise tests **function approximation**.

Tabular Q-learning is not suitable when the state is continuous or very large.  
DQN replaces the Q-table with a neural network:


$$
Q(s,a;w) \approx Q^*(s,a)
$$


Within the scope of the slides, DQN needs:

1. Q-network
2. Epsilon-greedy
3. Experience replay
4. Fixed target network
5. MSE Bellman loss

## Requirements

Run DQN on CartPole and answer:

1. What is the replay buffer used for?
2. What is the target network used for?
3. How is the loss derived from the Bellman target?


In [ ]:

if torch is not None:
    class QNetwork(nn.Module):
        def __init__(self, state_dim, action_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(state_dim, 128),
                nn.ReLU(),
                nn.Linear(128, 128),
                nn.ReLU(),
                nn.Linear(128, action_dim)
            )

        def forward(self, x):
            return self.net(x)

    Transition = namedtuple("Transition", ["state", "action", "reward", "next_state", "done"])

    class ReplayBuffer:
        def __init__(self, capacity=50_000):
            self.buffer = deque(maxlen=capacity)

        def push(self, *args):
            self.buffer.append(Transition(*args))

        def sample(self, batch_size):
            batch = random.sample(self.buffer, batch_size)
            return Transition(*zip(*batch))

        def __len__(self):
            return len(self.buffer)

    def select_action(policy_net, state, epsilon, action_dim, device):
        if random.random() < epsilon:
            return random.randrange(action_dim)

        with torch.no_grad():
            state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            q_values = policy_net(state_t)
            return int(q_values.argmax(dim=1).item())


In [ ]:

def train_dqn_cartpole(
    episodes=150,
    batch_size=64,
    gamma=0.99,
    lr=1e-3,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.995,
    target_update_every=10
):
    if torch is None:
        raise RuntimeError("Install torch to run the DQN exercise.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    env = gym.make("CartPole-v1")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy_net = QNetwork(state_dim, action_dim).to(device)
    target_net = QNetwork(state_dim, action_dim).to(device)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    memory = ReplayBuffer()

    epsilon = epsilon_start
    returns = []

    for ep in range(episodes):
        state = reset_env(env)
        total_reward = 0

        for t in range(500):
            action = select_action(policy_net, state, epsilon, action_dim, device)
            next_state, reward, done, info = step_env(env, action)

            memory.push(state, action, reward, next_state, done)

            state = next_state
            total_reward += reward

            if len(memory) >= batch_size:
                batch = memory.sample(batch_size)

                states = torch.tensor(np.array(batch.state), dtype=torch.float32, device=device)
                actions = torch.tensor(batch.action, dtype=torch.long, device=device).unsqueeze(1)
                rewards = torch.tensor(batch.reward, dtype=torch.float32, device=device).unsqueeze(1)
                next_states = torch.tensor(np.array(batch.next_state), dtype=torch.float32, device=device)
                dones = torch.tensor(batch.done, dtype=torch.float32, device=device).unsqueeze(1)

                q_sa = policy_net(states).gather(1, actions)

                with torch.no_grad():
                    next_q = target_net(next_states).max(dim=1, keepdim=True)[0]
                    target = rewards + gamma * next_q * (1 - dones)

                loss = F.mse_loss(q_sa, target)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            if done:
                break

        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        returns.append(total_reward)

        if (ep + 1) % target_update_every == 0:
            target_net.load_state_dict(policy_net.state_dict())

        if (ep + 1) % 25 == 0:
            print(f"Episode {ep+1}, average return={np.mean(returns[-25:]):.1f}, epsilon={epsilon:.3f}")

    env.close()
    return policy_net, returns

# Run if torch is available:
# policy_net, dqn_returns = train_dqn_cartpole(episodes=150)
# plt.figure()
# plt.plot(dqn_returns)
# plt.title("DQN on CartPole")
# plt.xlabel("Episode")
# plt.ylabel("Return")
# plt.show()



# Exercise 6 - Whiteboard algorithm adaptation + REINFORCE code

This exercise tests **policy-based methods**.

Policy Gradient learns directly:

$$
\pi_\theta(a \mid s)
$$

REINFORCE loss:

$$
L = -\sum_t \log \pi_\theta(a_t \mid s_t)G_t
$$

Actor-Critic adds a critic \(V(s)\) and uses advantage:

$$
A_t = G_t - V(s_t)
$$

## Part A - Whiteboard: run a policy-gradient update by hand

Given one episode with 3 steps:

| t | chosen action | \(\log \pi(a_t \mid s_t)\) | discounted return \(G_t\) |
|---|---|---:|---:|
| 0 | RIGHT | -0.7 | 1.0 |
| 1 | RIGHT | -0.5 | 1.0 |
| 2 | DOWN | -1.2 | 1.0 |

REINFORCE loss:

$$
L = -\sum_t \log \pi(a_t \mid s_t)G_t
$$

Requirements:

1. Compute the loss.
2. Since \(G_t > 0\), should the probabilities of the chosen actions increase or decrease?
3. If the episode has a negative return, how would the update direction change?

## Part B - Algorithm adaptation

Assume rewards are very sparse: the agent only receives `+1` at the end of the episode if it wins, and `0` otherwise.

Students should propose 2 adjustments within the scope of the slides:

1. Should discounted returns be used? Why?
2. Should a critic/baseline be added? Why?
3. Should exploration be maintained? Why?


## Student answer section

Write your solution / calculations here.


In [ ]:

if torch is not None:
    class PolicyNetwork(nn.Module):
        def __init__(self, state_dim, action_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(state_dim, 128),
                nn.ReLU(),
                nn.Linear(128, action_dim)
            )

        def forward(self, x):
            logits = self.net(x)
            return torch.distributions.Categorical(logits=logits)

    def compute_returns(rewards, gamma=0.99):
        returns = []
        G = 0

        for r in reversed(rewards):
            G = r + gamma * G
            returns.append(G)

        returns.reverse()
        return returns


In [ ]:

def train_reinforce_cartpole(episodes=300, gamma=0.99, lr=1e-2):
    if torch is None:
        raise RuntimeError("Install torch to run REINFORCE.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    env = gym.make("CartPole-v1")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy = PolicyNetwork(state_dim, action_dim).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    episode_returns = []

    for ep in range(episodes):
        state = reset_env(env)

        log_probs = []
        rewards = []
        total_reward = 0

        for t in range(500):
            state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            dist = policy(state_t)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))

            next_state, reward, done, info = step_env(env, int(action.item()))

            rewards.append(reward)
            total_reward += reward
            state = next_state

            if done:
                break

        returns = torch.tensor(compute_returns(rewards, gamma), dtype=torch.float32, device=device)

        # Normalize to reduce variance.
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        loss = 0
        for log_prob, G in zip(log_probs, returns):
            loss += -log_prob * G

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        episode_returns.append(total_reward)

        if (ep + 1) % 50 == 0:
            print(f"Episode {ep+1}, average return={np.mean(episode_returns[-50:]):.1f}")

    env.close()
    return policy, episode_returns

# Run if torch is available:
# pg_policy, pg_returns = train_reinforce_cartpole(episodes=300)
# plt.figure()
# plt.plot(pg_returns)
# plt.title("REINFORCE on CartPole")
# plt.xlabel("Episode")
# plt.ylabel("Return")
# plt.show()



# 6-point grading rubric

| Exercise | Criteria | Points |
|---|---|---:|
| 1 | Correctly runs the trajectory, computes returns, and adjusts the policy to avoid the hole | 1 |
| 2 | Correctly implements Value Iteration and extracts the policy | 1 |
| 3 | Correctly runs Bellman backup / Q backup and distinguishes synchronous from in-place updates | 1 |
| 4 | Correctly implements Q-learning update + epsilon-greedy and can read the policy | 1 |
| 5 | Correctly runs/explains DQN, replay buffer, target network, and Bellman loss | 1 |
| 6 | Computes REINFORCE loss, explains the update direction, and proposes critic/exploration for sparse rewards | 1 |

Total: **6 points**



# Quick review checklist before submission

You should be able to do these, not just state definitions:

1. Simulate a trajectory `state -> action -> reward -> next_state`.
2. Compute discounted returns from a reward sequence.
3. Run one Bellman backup by hand.
4. Choose an action from \(V(s)\) using one-step lookahead.
5. Choose an action from \(Q(s,a)\) using `argmax`.
6. Update one Q-learning step with concrete numbers.
7. Explain epsilon-greedy when the learned policy is currently wrong.
8. Explain the replay buffer and target network in DQN.
9. Compute REINFORCE loss from log-probabilities and returns.
10. Know when to add a critic/baseline to reduce variance.
